# Notebook 01: Wait-Time Distribution Model
## TransPlan Validation & Reproducibility Pack (Phase 4 M5)

This notebook documents the **log-normal wait-time distribution model** that underlies TransPlan's Monte Carlo simulation engine. It covers:

1. **Model specification** — why log-normal, how parameters are derived from SRTR data
2. **Parameter visualization** — national medians, blood type multipliers, city factors, clinical adjusters
3. **Distribution shapes** — PDF/CDF curves for representative patient profiles
4. **SRTR comparison** — model predictions vs. empirical SRTR center-level percentiles
5. **Sensitivity analysis** — which parameters move the distribution most

### Data Source
SRTR Program-Specific Reports (PSR), January 2025 release. Table B10 provides center-level wait-time percentiles (P5/P10/P25/P50/P75) which are fitted to log-normal distributions via method-of-moments on log-transformed percentile pairs.

### References
- SRTR PSR Technical Methods: https://www.srtr.org/about-the-data/technical-methods-for-the-program-specific-reports/
- Log-normal transplant wait times: Wolfe et al., *Transplantation*, 2010

In [1]:
# --- Setup: path configuration, imports, reproducibility seed ---
import sys, os, json, hashlib
from pathlib import Path

# Add backend/ to Python path so we can import TransPlan services
REPO_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT / "backend"))
os.chdir(REPO_ROOT / "backend")  # So config.py DATA_DIR resolves correctly

import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Reproducibility
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120
matplotlib.use("Agg")  # Non-interactive backend for headless execution

# Log data file hash for reproducibility
data_path = REPO_ROOT / "data" / "wait-time-distributions.json"
file_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()[:12]
print(f"Data file: {data_path.name}")
print(f"SHA-256 (first 12): {file_hash}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}, SciPy: {scipy.__version__}, Pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}, Seaborn: {sns.__version__}")
print(f"RNG seed: {RNG_SEED}")

Data file: wait-time-distributions.json
SHA-256 (first 12): 289de3ad599a
Python: 3.14.3
NumPy: 2.4.2, SciPy: 1.17.1, Pandas: 2.3.3
Matplotlib: 3.10.8, Seaborn: 0.13.2
RNG seed: 42


## 1. Model Specification

TransPlan models transplant wait times as **log-normal distributions**:

$$T \sim \text{LogNormal}(\mu, \sigma)$$

where:
- $\mu = \ln(\text{adjusted\_median})$ (location parameter)
- $\sigma$ = shape parameter (controls spread/skewness)

The **adjusted median** is computed as:

$$\text{median} = \text{national\_median} \times \text{blood\_type\_mult} \times \text{city\_factor} \times \text{clinical\_mult}$$

### Why log-normal?

1. **Empirical fit**: SRTR wait-time percentiles are well-approximated by log-normal (right-skewed, non-negative)
2. **Clinical interpretation**: the median is the "typical" wait; the log-sigma controls how much variability exists
3. **Multiplicative factors**: blood type, city, and clinical adjusters compose naturally as multiplicative shifts to the median

### SciPy parameterization

`scipy.stats.lognorm(s=sigma, scale=median)` produces a distribution with:
- Median = `scale` (since $e^{\mu} = \text{scale}$ and median of log-normal = $e^{\mu}$)
- Shape parameter `s` = $\sigma$ (standard deviation of the underlying normal)

In [2]:
# Load raw distribution parameters
with open(REPO_ROOT / "data" / "wait-time-distributions.json") as f:
    dist_data = json.load(f)

# Extract organ parameters into a clean table
organs = ["kidney", "liver", "heart", "lung", "pancreas", "intestine"]
organ_params = []
for organ in organs:
    p = dist_data[organ]
    organ_params.append({
        "Organ": organ.title(),
        "National Median (months)": p["national_median_months"],
        "Log-Sigma (σ)": p["log_sigma"],
        "Mean (months)": round(p["national_median_months"] * np.exp(p["log_sigma"]**2 / 2), 1),
        "P75 (months)": round(stats.lognorm(s=p["log_sigma"], scale=p["national_median_months"]).ppf(0.75), 1),
        "P95 (months)": round(stats.lognorm(s=p["log_sigma"], scale=p["national_median_months"]).ppf(0.95), 1),
    })

params_df = pd.DataFrame(organ_params)
print("=== National Wait-Time Distribution Parameters (SRTR January 2025) ===\n")
print(params_df.to_string(index=False))

=== National Wait-Time Distribution Parameters (SRTR January 2025) ===

    Organ  National Median (months)  Log-Sigma (σ)  Mean (months)  P75 (months)  P95 (months)
   Kidney                      27.4           1.20           56.3          61.6         197.2
    Liver                       4.6           1.20            9.5          10.3          33.1
    Heart                       2.2           1.20            4.5           4.9          15.8
     Lung                       1.4           1.14            2.7           3.0           9.1
 Pancreas                      22.8           0.80           31.4          39.1          85.0
Intestine                      11.6           1.20           23.8          26.1          83.5


## 2. National Wait-Time PDFs by Organ

The probability density functions show dramatically different wait-time profiles across organs. Kidney patients face the longest waits (median ~27 months), while lung patients benefit from the LAS allocation system (median ~1.4 months).

In [3]:
# Plot national PDF curves for all 6 organs
FIGURES_DIR = REPO_ROOT / "notebooks" / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors = sns.color_palette("deep", 6)

for idx, organ in enumerate(organs):
    ax = axes[idx // 3, idx % 3]
    p = dist_data[organ]
    median = p["national_median_months"]
    sigma = p["log_sigma"]
    
    dist = stats.lognorm(s=sigma, scale=median)
    
    # X range: 0 to 99th percentile
    x_max = dist.ppf(0.99)
    x = np.linspace(0.01, x_max, 500)
    
    ax.fill_between(x, dist.pdf(x), alpha=0.3, color=colors[idx])
    ax.plot(x, dist.pdf(x), color=colors[idx], linewidth=2)
    
    # Mark median
    ax.axvline(median, color=colors[idx], linestyle="--", alpha=0.7, linewidth=1.5)
    ax.text(median, ax.get_ylim()[1] * 0.02, f"  median={median}mo", 
            fontsize=9, color=colors[idx], fontweight="bold", va="bottom")
    
    ax.set_title(f"{organ.title()}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Wait Time (months)")
    ax.set_ylabel("Probability Density")
    ax.set_xlim(0, x_max)

fig.suptitle("National Wait-Time Distributions by Organ Type\n(Log-Normal Model, SRTR January 2025)", 
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-organ-pdfs.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/01-organ-pdfs.png")

Saved: figures/01-organ-pdfs.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/689463878.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Blood Type Multipliers

Blood type significantly affects wait times due to ABO-compatibility matching rules. Type O patients wait longest (larger recipient pool competing for compatible organs), while AB patients wait shortest (universal recipients).

The multipliers are applied multiplicatively to the national median: a multiplier of 1.3 means 30% longer median wait.

In [4]:
# Blood type multiplier heatmap
blood_types = ["O+", "O-", "A+", "A-", "B+", "B-", "AB+", "AB-"]
bt_matrix = []

for organ in organs:
    mults = dist_data[organ].get("blood_type_multipliers", {})
    row = [mults.get(bt, 1.0) for bt in blood_types]
    bt_matrix.append(row)

bt_df = pd.DataFrame(bt_matrix, index=[o.title() for o in organs], columns=blood_types)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(bt_df, annot=True, fmt=".2f", cmap="RdYlGn_r", center=1.0,
            linewidths=1, linecolor="white", ax=ax,
            cbar_kws={"label": "Wait Time Multiplier (1.0 = national avg)"})
ax.set_title("Blood Type Wait-Time Multipliers by Organ\n(>1.0 = longer wait, <1.0 = shorter wait)", 
             fontsize=14, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-blood-type-heatmap.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/01-blood-type-heatmap.png")

Saved: figures/01-blood-type-heatmap.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/1559779236.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. City Wait-Time Factors

Each of the 22 TransPlan cities has a multiplicative factor reflecting its transplant center's historical wait times relative to the national average. Factors < 1.0 indicate faster-than-average centers.

In [5]:
# City wait-time factors bar chart
city_factors = dist_data.get("city_wait_time_factors", {})
city_factors = {k: v for k, v in city_factors.items() if k != "_notes"}

# Sort by factor value
sorted_cities = sorted(city_factors.items(), key=lambda x: x[1])
cities_sorted = [c[0] for c in sorted_cities]
factors_sorted = [c[1] for c in sorted_cities]

# Color: green for < 1.0 (faster), red for > 1.0 (slower)
bar_colors = ["#2ca02c" if f < 1.0 else "#d62728" if f > 1.0 else "#666666" for f in factors_sorted]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(cities_sorted, factors_sorted, color=bar_colors, edgecolor="white", linewidth=0.5)

# Reference line at 1.0
ax.axvline(1.0, color="black", linestyle="-", linewidth=1.5, alpha=0.7)

# Labels on bars
for bar, val in zip(bars, factors_sorted):
    offset = 0.02 if val < 1.0 else -0.02
    ha = "left" if val < 1.0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, f"{val:.2f}", 
            va="center", ha=ha, fontsize=9, fontweight="bold")

ax.set_xlabel("City Wait-Time Factor (1.0 = national average)", fontsize=12)
ax.set_title("TransPlan City Wait-Time Factors (22 Centers)\nGreen = faster than average, Red = slower than average", 
             fontsize=14, fontweight="bold")
ax.set_xlim(min(factors_sorted) - 0.15, max(factors_sorted) + 0.15)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-city-factors.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved: figures/01-city-factors.png")
print(f"\nFastest: {cities_sorted[0]} ({factors_sorted[0]:.2f}x)")
print(f"Slowest: {cities_sorted[-1]} ({factors_sorted[-1]:.2f}x)")
print(f"Range: {min(factors_sorted):.2f} - {max(factors_sorted):.2f}")

Saved: figures/01-city-factors.png

Fastest: Madison (0.51x)
Slowest: San Francisco (2.12x)
Range: 0.51 - 2.12


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/2653554042.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Clinical Multipliers (Organ-Specific)

Three organs have clinical-score-dependent wait-time adjusters:

- **Kidney**: cPRA (Calculated Panel Reactive Antibody) — highly sensitized patients (cPRA > 98) wait dramatically longer
- **Liver**: MELD score — sicker patients (high MELD) are prioritized and wait shorter
- **Lung**: LAS (Lung Allocation Score) — higher urgency scores lead to faster allocation

In [6]:
# Clinical multiplier visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Kidney: cPRA ---
cpra_mults = dist_data["kidney"]["clinical_multipliers"]["cpra"]
cpra_ranges = list(cpra_mults.keys())
cpra_vals = list(cpra_mults.values())
kidney_median = dist_data["kidney"]["national_median_months"]

ax = axes[0]
bars = ax.bar(cpra_ranges, cpra_vals, color=sns.color_palette("Reds", len(cpra_ranges)), edgecolor="white")
for bar, val in zip(bars, cpra_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f"{val}x\n({val * kidney_median:.0f}mo)", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Kidney: cPRA Multiplier", fontsize=13, fontweight="bold")
ax.set_xlabel("cPRA Range")
ax.set_ylabel("Wait Time Multiplier")
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)

# --- Liver: MELD ---
meld_mults = dist_data["liver"]["clinical_multipliers"]["meld"]
meld_ranges = list(meld_mults.keys())
meld_vals = list(meld_mults.values())
liver_median = dist_data["liver"]["national_median_months"]

ax = axes[1]
bars = ax.bar(meld_ranges, meld_vals, color=sns.color_palette("Greens_r", len(meld_ranges)), edgecolor="white")
for bar, val in zip(bars, meld_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
            f"{val}x\n({val * liver_median:.1f}mo)", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Liver: MELD Multiplier", fontsize=13, fontweight="bold")
ax.set_xlabel("MELD Range")
ax.set_ylabel("Wait Time Multiplier")
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)

# --- Lung: LAS ---
las_mults = dist_data["lung"]["clinical_multipliers"]["las"]
las_ranges = list(las_mults.keys())
las_vals = list(las_mults.values())
lung_median = dist_data["lung"]["national_median_months"]

ax = axes[2]
bars = ax.bar(las_ranges, las_vals, color=sns.color_palette("Blues_r", len(las_ranges)), edgecolor="white")
for bar, val in zip(bars, las_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
            f"{val}x\n({val * lung_median:.1f}mo)", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Lung: LAS Multiplier", fontsize=13, fontweight="bold")
ax.set_xlabel("LAS Range")
ax.set_ylabel("Wait Time Multiplier")
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5)

fig.suptitle("Organ-Specific Clinical Score Multipliers\n(Applied multiplicatively to national median)", 
             fontsize=15, fontweight="bold", y=1.04)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-clinical-multipliers.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/01-clinical-multipliers.png")

Saved: figures/01-clinical-multipliers.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/924353384.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Composed Distribution: Full Patient Profile

Now we demonstrate how all multipliers compose to form a specific patient's wait-time distribution. We'll compare several realistic kidney patient profiles to show how blood type, city, and cPRA interact.

In [7]:
# Import the actual TransPlan distribution service to ensure we match production behavior
from services.distributions import get_wait_time_distribution

# Define representative patient profiles
profiles = [
    {"label": "Kidney O+ / Cleveland / cPRA 30\n(typical hard case)", 
     "organ": "kidney", "blood_type": "O+", "city": "Cleveland", "cpra": 30},
    {"label": "Kidney AB+ / Pittsburgh / cPRA 10\n(best case)", 
     "organ": "kidney", "blood_type": "AB+", "city": "Pittsburgh", "cpra": 10},
    {"label": "Kidney O+ / New York / cPRA 99\n(hardest case)", 
     "organ": "kidney", "blood_type": "O+", "city": "New York", "cpra": 99},
    {"label": "Kidney A+ / national avg / cPRA 0\n(baseline reference)", 
     "organ": "kidney", "blood_type": "A+", "city": "Chicago", "cpra": 0},
]

fig, ax = plt.subplots(figsize=(14, 7))
colors = ["#d62728", "#2ca02c", "#7f7f7f", "#1f77b4"]

for i, prof in enumerate(profiles):
    dist = get_wait_time_distribution(
        organ=prof["organ"], blood_type=prof["blood_type"],
        city=prof["city"], cpra=prof.get("cpra"),
        meld=prof.get("meld"), las=prof.get("las"),
    )
    
    x_max = min(dist.ppf(0.99), 200)  # Cap at 200 months for readability
    x = np.linspace(0.01, x_max, 500)
    
    ax.plot(x, dist.pdf(x), color=colors[i], linewidth=2.5, label=prof["label"])
    ax.fill_between(x, dist.pdf(x), alpha=0.1, color=colors[i])
    
    # Mark median
    median = dist.median()
    ax.axvline(median, color=colors[i], linestyle=":", alpha=0.5, linewidth=1)
    
    print(f"{prof['label'].split(chr(10))[0]}:")
    print(f"  Median: {median:.1f} months, Mean: {dist.mean():.1f} months")
    print(f"  P(wait < 12mo): {dist.cdf(12):.1%}, P(wait < 24mo): {dist.cdf(24):.1%}")
    print()

ax.set_xlabel("Wait Time (months)", fontsize=12)
ax.set_ylabel("Probability Density", fontsize=12)
ax.set_title("Kidney Wait-Time Distributions: Four Patient Profiles\n(Showing how blood type, city, and cPRA shift the distribution)", 
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=10, framealpha=0.9)
ax.set_xlim(0, 150)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-kidney-profiles.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/01-kidney-profiles.png")

Kidney O+ / Cleveland / cPRA 30:
  Median: 66.8 months, Mean: 137.2 months
  P(wait < 12mo): 7.6%, P(wait < 24mo): 19.7%

Kidney AB+ / Pittsburgh / cPRA 10:
  Median: 21.4 months, Mean: 44.0 months
  P(wait < 12mo): 31.5%, P(wait < 24mo): 53.8%

Kidney O+ / New York / cPRA 99:
  Median: 190.6 months, Mean: 391.5 months
  P(wait < 12mo): 1.1%, P(wait < 24mo): 4.2%

Kidney A+ / national avg / cPRA 0:
  Median: 19.0 months, Mean: 39.0 months
  P(wait < 12mo): 35.1%, P(wait < 24mo): 57.7%



Saved: figures/01-kidney-profiles.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/687784599.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. CDF Comparison Across All 22 Cities

The cumulative distribution function P(wait ≤ t) shows the probability of receiving a transplant by time t, **ignoring competing risks** (mortality, delisting). This is the "pure wait-time" view — the full competing-risks model is documented in Notebook 02.

We plot CDFs for kidney O+ patients across all 22 cities to show the geographic variation.

In [8]:
# CDF curves for kidney O+ across all 22 cities
all_cities = [
    "Pittsburgh", "Baltimore", "Philadelphia", "New York", "Minneapolis",
    "Madison", "Chicago", "Cleveland", "St. Louis", "Indianapolis",
    "Omaha", "Rochester", "Nashville", "Durham", "Miami",
    "Dallas", "Houston", "Portland", "Seattle", "San Francisco",
    "Los Angeles", "Palo Alto",
]

fig, ax = plt.subplots(figsize=(14, 8))
t = np.linspace(0, 72, 500)  # 0 to 6 years

# Compute CDFs and sort by 24-month probability
city_cdfs = []
for city in all_cities:
    dist = get_wait_time_distribution("kidney", "O+", city, cpra=30)
    cdf_24 = dist.cdf(24)
    city_cdfs.append((city, dist, cdf_24))

city_cdfs.sort(key=lambda x: x[2], reverse=True)

# Color map: best (green) to worst (red)
cmap = plt.cm.RdYlGn
norm_vals = np.linspace(0, 1, len(city_cdfs))

for i, (city, dist, cdf_24) in enumerate(city_cdfs):
    color = cmap(norm_vals[i])
    alpha = 0.9 if i < 3 or i >= len(city_cdfs) - 3 else 0.4
    lw = 2.5 if i < 3 or i >= len(city_cdfs) - 3 else 1.0
    label = f"{city} ({cdf_24:.0%})" if i < 3 or i >= len(city_cdfs) - 3 else None
    ax.plot(t, dist.cdf(t), color=color, alpha=alpha, linewidth=lw, label=label)

# Reference lines
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.3, linewidth=1)
ax.axvline(24, color="gray", linestyle="--", alpha=0.3, linewidth=1)
ax.text(24.5, 0.02, "24 months", fontsize=9, color="gray")
ax.text(1, 0.51, "50% probability", fontsize=9, color="gray")

ax.set_xlabel("Wait Time (months)", fontsize=12)
ax.set_ylabel("P(transplant ≤ t)", fontsize=12)
ax.set_title("Kidney O+ (cPRA 30) CDF Across 22 Cities\n(Top 3 and bottom 3 cities highlighted)", 
             fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10, title="City (P ≤ 24mo)", framealpha=0.9)
ax.set_xlim(0, 72)
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-city-cdfs-kidney.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/01-city-cdfs-kidney.png")

Saved: figures/01-city-cdfs-kidney.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/3012366047.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Sensitivity: Which Parameter Matters Most?

We decompose the variance contribution of each parameter type by computing the median wait time at each level while holding others at reference values. This shows the relative importance of blood type vs. city vs. clinical score for kidney patients.

In [9]:
# Sensitivity: tornado-style chart for kidney wait-time median
kidney_base_median = dist_data["kidney"]["national_median_months"]
reference_city = "Chicago"
reference_bt = "A+"
reference_cpra = 0

# Base case
base_dist = get_wait_time_distribution("kidney", reference_bt, reference_city, cpra=reference_cpra)
base_median = base_dist.median()

# Vary blood type (hold city=Chicago, cPRA=0)
bt_medians = {}
for bt in blood_types:
    d = get_wait_time_distribution("kidney", bt, reference_city, cpra=reference_cpra)
    bt_medians[bt] = d.median()

# Vary city (hold bt=A+, cPRA=0)
city_medians = {}
for city in all_cities:
    d = get_wait_time_distribution("kidney", reference_bt, city, cpra=reference_cpra)
    city_medians[city] = d.median()

# Vary cPRA (hold bt=A+, city=Chicago)
cpra_test_values = [0, 10, 30, 50, 80, 90, 95, 99, 100]
cpra_medians = {}
for cpra in cpra_test_values:
    d = get_wait_time_distribution("kidney", reference_bt, reference_city, cpra=cpra)
    cpra_medians[cpra] = d.median()

# Summary
print(f"=== Kidney Median Wait-Time Sensitivity (reference: {reference_bt}, {reference_city}, cPRA={reference_cpra}) ===\n")
print(f"Reference median: {base_median:.1f} months\n")
print(f"Blood type range:  {min(bt_medians.values()):.1f} - {max(bt_medians.values()):.1f} months "
      f"(ratio: {max(bt_medians.values())/min(bt_medians.values()):.1f}x)")
print(f"City range:        {min(city_medians.values()):.1f} - {max(city_medians.values()):.1f} months "
      f"(ratio: {max(city_medians.values())/min(city_medians.values()):.1f}x)")
print(f"cPRA range:        {min(cpra_medians.values()):.1f} - {max(cpra_medians.values()):.1f} months "
      f"(ratio: {max(cpra_medians.values())/min(cpra_medians.values()):.1f}x)")

# Tornado chart
fig, ax = plt.subplots(figsize=(12, 4))

params = ["Blood Type\n(O- vs AB+)", "City Factor\n(worst vs best)", "cPRA Score\n(0 vs 99+)"]
lo_vals = [min(bt_medians.values()), min(city_medians.values()), min(cpra_medians.values())]
hi_vals = [max(bt_medians.values()), max(city_medians.values()), max(cpra_medians.values())]

y_pos = range(len(params))
for i, (param, lo, hi) in enumerate(zip(params, lo_vals, hi_vals)):
    ax.barh(i, hi - base_median, left=base_median, color="#d62728", alpha=0.7, height=0.5)
    ax.barh(i, lo - base_median, left=base_median, color="#2ca02c", alpha=0.7, height=0.5)
    ax.text(hi + 0.5, i, f"{hi:.0f}mo", va="center", fontsize=10, fontweight="bold", color="#d62728")
    ax.text(lo - 0.5, i, f"{lo:.0f}mo", va="center", fontsize=10, fontweight="bold", color="#2ca02c", ha="right")

ax.set_yticks(list(y_pos))
ax.set_yticklabels(params, fontsize=11)
ax.axvline(base_median, color="black", linewidth=2)
ax.set_xlabel("Median Wait Time (months)", fontsize=12)
ax.set_title(f"Kidney Wait-Time Sensitivity Tornado\n(Reference: {reference_bt}, {reference_city}, cPRA={reference_cpra})", 
             fontsize=14, fontweight="bold")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-sensitivity-tornado.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/01-sensitivity-tornado.png")

=== Kidney Median Wait-Time Sensitivity (reference: A+, Chicago, cPRA=0) ===

Reference median: 19.0 months

Blood type range:  11.6 - 29.5 months (ratio: 2.5x)
City range:        12.6 - 52.3 months (ratio: 4.2x)
cPRA range:        19.0 - 94.9 months (ratio: 5.0x)


Saved: figures/01-sensitivity-tornado.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/3074228870.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Monte Carlo Sampling Validation

We verify that drawing random samples from the log-normal distribution reproduces the theoretical percentiles. This confirms that `scipy.stats.lognorm` behaves as expected and that our parameterization is correct.

In [10]:
# Monte Carlo sampling vs theoretical distribution
N_SAMPLES = 50_000

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, organ in enumerate(organs):
    ax = axes[idx // 3, idx % 3]
    p = dist_data[organ]
    median = p["national_median_months"]
    sigma = p["log_sigma"]
    
    dist = stats.lognorm(s=sigma, scale=median)
    
    # Draw samples
    samples = dist.rvs(size=N_SAMPLES, random_state=rng)
    
    # Histogram of samples
    x_max = dist.ppf(0.97)
    bins = np.linspace(0, x_max, 60)
    ax.hist(samples, bins=bins, density=True, alpha=0.4, color="steelblue", 
            edgecolor="white", label=f"MC samples (n={N_SAMPLES:,})")
    
    # Theoretical PDF
    x = np.linspace(0.01, x_max, 500)
    ax.plot(x, dist.pdf(x), "r-", linewidth=2, label="Theoretical PDF")
    
    # Percentile comparison
    theoretical_pcts = [dist.ppf(q) for q in [0.25, 0.50, 0.75]]
    empirical_pcts = [np.percentile(samples, q) for q in [25, 50, 75]]
    
    ax.set_title(f"{organ.title()}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Wait Time (months)")
    ax.set_xlim(0, x_max)
    
    # Annotate percentile match
    pct_text = (f"P25: {theoretical_pcts[0]:.1f} vs {empirical_pcts[0]:.1f}\n"
                f"P50: {theoretical_pcts[1]:.1f} vs {empirical_pcts[1]:.1f}\n"
                f"P75: {theoretical_pcts[2]:.1f} vs {empirical_pcts[2]:.1f}")
    ax.text(0.97, 0.97, pct_text, transform=ax.transAxes, fontsize=8,
            va="top", ha="right", bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.8))
    
    if idx == 0:
        ax.legend(loc="upper left", fontsize=9)

fig.suptitle(f"Monte Carlo Sampling Validation (n={N_SAMPLES:,} draws per organ)\n"
             "Histogram = samples, Red line = theoretical PDF, Inset = percentile comparison",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-mc-validation.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/01-mc-validation.png")

Saved: figures/01-mc-validation.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/1806914553.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Cross-Organ Comparison: 24-Month Transplant Probability

A summary view: for a "typical" patient (A+ blood, cPRA 0 / MELD 20 / LAS 50 as appropriate), what is P(transplant ≤ 24 months) for each organ across all cities?

In [11]:
# Cross-organ 24-month probability matrix
organ_configs = {
    "kidney": {"blood_type": "A+", "cpra": 0},
    "liver": {"blood_type": "A+", "meld": 20},
    "heart": {"blood_type": "A+"},
    "lung": {"blood_type": "A+", "las": 50.0},
    "pancreas": {"blood_type": "A+"},
    "intestine": {"blood_type": "A+"},
}

p24_matrix = []
for city in all_cities:
    row = {"City": city}
    for organ in organs:
        cfg = organ_configs[organ]
        dist = get_wait_time_distribution(organ, cfg["blood_type"], city,
                                           cpra=cfg.get("cpra"), meld=cfg.get("meld"), las=cfg.get("las"))
        row[organ.title()] = dist.cdf(24)
    p24_matrix.append(row)

p24_df = pd.DataFrame(p24_matrix).set_index("City")

# Display as heatmap
fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(p24_df, annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1,
            linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label": "P(transplant ≤ 24 months)"})

ax.set_title("P(Transplant ≤ 24 Months) by City and Organ\n(Typical patient: A+ blood, standard clinical scores, wait-time only)", 
             fontsize=13, fontweight="bold")
ax.set_ylabel("")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "01-cross-organ-p24.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: figures/01-cross-organ-p24.png")

Saved: figures/01-cross-organ-p24.png


/var/folders/2v/mppjqytd61b5lt_ymg09qff80000gn/T/ipykernel_58012/3973092393.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Summary & Known Limitations

### Key Findings
- **Kidney** has the widest geographic variation (city factors span ~0.7x to ~1.3x) and the strongest clinical modifier (cPRA 99+ multiplies wait by 5x)
- **Liver** MELD-based allocation creates an inverted relationship: sicker patients are prioritized → shorter waits
- **Lung** LAS system produces the shortest waits and least geographic variation
- **Blood type** O consistently increases waits across all organs (1.1-1.4x), while AB shortens them (0.55-0.8x)

### Limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| Log-normal assumption | May not capture bimodal distributions (e.g., status 1A heart) | Could add mixture models in Phase 5 |
| Uniform σ across cities | All cities share the same log-sigma per organ | SRTR center-level σ fitting planned (L-049) |
| Blood type multipliers from literature | Not from SRTR center-level data (Table B10 doesn't stratify by blood type) | Cross-validate with OPTN blood-type-specific data |
| cPRA/MELD/LAS multipliers are stepped | Discrete ranges, not continuous functions | Smooth interpolation possible but adds complexity |
| No time-series updating | Parameters are static (January 2025 SRTR release) | Historical trends (Notebook 04) provide trajectory context |

### Next Notebook
**Notebook 02: Competing Risks Model** — adds mortality and delisting as competing events, showing how P(transplant) shrinks when patients can die or be delisted before receiving an organ.